# Titanic Exploratory Data Analysis

An in-depth analysis of the famous Titanic dataset to understand the factors that influenced passenger survival.

**Technologien:** `pandas`, `matplotlib`, `seaborn`

### Research Questions
- Which class (Pclass) had the highest survival rate?
- How did gender affect survival (the 'women and children first' principle)?
- Did age play a role — were younger passengers more likely to survive?
- Were there patterns related to ticket classes vs. embarkation ports?

In [ ]:
# ==============================================
# CHAPTER 1: DATA LOADING & INITIAL OVERVIEW
# ==============================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16

# Load dataset
URL = 'https://raw.githubusercontent.com/datasciencehandbook/notebooks/data/titanic.csv'
df = pd.read_csv(URL)

# Initial overview
print('=== DATA OVERVIEW ===')
print(f'Zeilen: {len(df)} | Spalten: {len(df.columns)}')
print()
print(df.head())
print()
print('=== DATA TYPES ===')
print(df.dtypes)
print()
print('=== STATISTICAL SUMMARY ===')
print(df.describe())

In [ ]:
# ==============================================
# KAPITEL 2: MISSING VALUES & DATENQUALITAT
# ==============================================

# Missing values per column
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

total_cells = len(df) * len(df.columns)
pct_missing = missing.sum() / total_cells * 100
print(f'\nTotal missing values: {missing.sum()} of {total_cells} ({pct_missing:.1f}%)')

# Visualisierung 1: Heatmap der fehlenden Werte
plt.figure(figsize=(12, 6))
missing_pct = df.isnull().sum() / len(df) * 100
ax = sns.barplot(
    x=missing_pct.index,
    y=missing_pct.values,
    palette='Oranges_r',
    edgecolor='.4'
)
ax.set_title('Missing Values per Feature (%)', fontsize=16, fontweight='bold')
ax.set_xlabel('Feature', fontsize=12)
ax.set_ylabel('Anteil (%)', fontsize=12)

# Werte auf den Balken anzeigen
for bar in ax.patches:
    if bar.get_height() > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2.,
            bar.get_height() * 0.95,
            f'{bar.get_height():.0f}%',
            ha='center', va='bottom', fontsize=10
        )

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================
# CHAPTER 3: SURVIVAL RATE BY CLASS & GENDER
# ==============================================

# Hauptfrage: Wer hat überlebt?
survived_count = (df['Survived'] == 1).sum()
total = len(df)
rate = df['Survived'].mean() * 100
print(f'=== SURVIVAL RATE (OVERALL) ===')
print(f'Survivors: {survived_count} / {total} ({rate:.1f}%)')

# Visualization 2: Survival by Class and Gender
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# A) Nach Pclass
survival_pc = df.groupby('Pclass')['Survived'].mean()
colors = ['#e74c3c', '#3498db', '#2ecc71']
bars = axes[0].bar(range(1, 4), survival_pc.values, color=colors, edgecolor='.3')
axes[0].set_title(Survival Rate by Passenger Class, fontsize=15, fontweight='bold')
axes[0].set_ylabel(Survival Rate, fontsize=12)
axes[0].set_xlabel(Class, fontsize=12)
axes[0].set_xticks([0.5, 1.5, 2.5])
axes[0].set_xticklabels([1st Class, 2nd Class, 3rd Class])

# Werte anzeigen
for bar, val in zip(bars, survival_pc.values):
    axes[0].text(
        bar.get_x() + bar.get_width()/2.,
        bar.get_height() + 0.01,
        f'{val:.0%}', ha='center', fontsize=10
    )

# B) By Gender
sex_surv = df.groupby('Sex')['Survived'].mean()
bars2 = axes[1].bar([Female, Male], sex_surv.values, color=['#e67e22', '#9b59b6'], edgecolor='.3')
axes[1].set_title(Survival Rate by Gender, fontsize=15, fontweight='bold')
axes[1].set_ylabel(Survival Rate, fontsize=12)

for bar, val in zip(bars2, sex_surv.values):
    axes[1].text(
        bar.get_x() + bar.get_width()/2.,
        bar.get_height() + 0.01,
        f'{val:.0%}', ha='center', fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================
# CHAPTER 4: AGE & SURVIVAL
# ==============================================

# Age distribution of survivors vs. non-survivors
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# A) Histogramm der Alter
df[df['Survived']==1]['Age'].dropna().hist(
    bins=30, alpha=0.7, label='Überlebt', ax=axes[0], color='#2ecc71', density=True
)
df[df['Survived']==0]['Age'].dropna().hist(
    bins=30, alpha=0.7, label='Gestorben', ax=axes[0], color='#e74c3c', density=True
)
axes[0].set_title(Age Distribution - Survived vs. Died, fontsize=15, fontweight='bold')
axes[0].set_xlabel(Age (years), fontsize=12)
axes[0].set_ylabel(Density, fontsize=12)
axes[0].legend()

# B) Survival rate by age groupn
age_bins = [0, 5, 13, 18, 35, 60, 100]
df_copy = df.copy()
df_copy['Age Group'] = pd.cut(
    df['Age'],
    bins=age_bins,
    labels=['0-5', '6-12', '13-18', '19-34', '35-59', '60+']
)
rate_by_age = df_copy.groupby('Age Group')['Survived'].mean().sort_index()

bars3 = axes[1].bar(
    range(len(rate_by_age)),
    rate_by_age.values,
    color=['#f39c12', '#e67e22', '#d35400', '#c0392b', '#8e44ad', '#2980b9'],
    edgecolor='.3'
)
axes[1].set_title('Survival rate by age group', fontsize=15, fontweight='bold')
axes[1].set_xlabel('Age Group', fontsize=12)
axes[1].set_ylabel(Survival Rate, fontsize=12)
axes[1].set_xticks(range(len(rate_by_age)))
labels = [f'{lb}' for lb in rate_by_age.index]
axes[1].set_xticklabels(labels, rotation=45, ha='right')

# Werte auf den Balken
for bar, val in zip(bars3, rate_by_age.values):
    axes[1].text(
        bar.get_x() + bar.get_width()/2.,
        bar.get_height() + 0.01,
        f'{val:.0%}', ha='center', fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================
# CHAPTER 5: FARE ANALYSIS BY CLASS
# ==============================================

# Visualisierung: Average ticket price per class
fig, ax = plt.subplots(figsize=(10, 6))

fare_by_class = df.groupby('Pclass')['Fare'].agg(['mean', 'Average', 'min', 'max'])
colors_fare = ['#e74c3c', '#f39c12', '#2ecc71']

# Balkendiagramm des Averagespreises
bars4 = ax.bar(
    range(1, 4), fare_by_class['mean'].values,
    color=colors_fare[:3], edgecolor='.3', label=Average
)
# Average line
ax.plot(
    [0.5, 1.5, 2.5],
    fare_by_class['Average'].values,
    'o-', color='#3498db', linewidth=2, label='Average'
)

ax.set_title(Ticket Price by Passenger Class, fontsize=15, fontweight='bold')
ax.set_ylabel(Price ($), fontsize=12)
ax.set_xlabel(Class, fontsize=12)
ax.set_xticks([0.5, 1.5, 2.5])
ax.set_xticklabels([1st Class, 2nd Class, 3rd Class])
ax.legend()

# Werte auf den Balken
for bar in bars4:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2.,
        height + 5,
        f'${height:.0f}',
        ha='center', fontsize=10, fontweight='bold'
    )

# Statistische Tabelle
print('=== TICKET PRICES BY CLASS ===')
for pc in range(1, 4):
    subset = df[df['Pclass'] == pc]
    print(f'\nPclass {pc}:')
    print(f'  Anzahl: {len(subset)}')
    print(f'  Average: ${subset["Fare"].mean():.2f}')
    print(f'  Average:     ${subset["Fare"].median():.2f}')
    print(f'  Min / Max:  {subset["Fare"].min():.2f} / {subset["Fare"].max():.2f}')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================
# CHAPTER 6: CORRELATION HEATMAP
# ==============================================

# Only numeric columns for correlation
numeric_cols = df.select_dtypes(include=['float64', 'int64'])
corr = numeric_cols.corr()

# Heatmap
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    center=0,
    linewidths=.5,
    square=True,
    fmt='.2f',
    xticklabels=True,
    yticklabels=True
)
ax.set_title(Correlation Matrix - Numerical Features, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Interpretation
print('=== CORRELATION INTERPRETATION ===')
surv_corr = corr['Survived'].drop('Survived').sort_values(ascending=False)
print(surv_corr.to_string())

# ZUSAMMENFASSUNG & KEY FINDINGS

### Key Findings

1. **Class = Survival Chance**
   Pclass has the greatest impact on survival. 1st class passengers had a significantly higher survival rate (~63%) compared to 3rd class (~24%).

2. **Gender plays a significant role**
   Frauen überlebten mit ~75% viel häufiger als Männer (~19%) — das "Frauen und Kinder zuerst"-Prinzip wurde befolgt.

3. **Younger passengers had better chances**
   Children and teenagers survived at higher rates.

4. **Fare correlates strongly with class**
   Higher ticket prices often correlated with better Pclass - those who paid more survived at higher rates.

5. **Cabin has ~80% missing values**
   Cabin information is unavailable for most passengers.

6. **Embarkation Port (C = Cherbourg) correlates with higher survival rate**
   Passengers who embarked from Cherbourg had a slightly higher survival rate.